# HNL mu-pi track-cut thresholds

Tunes the thresholds of `ana.TRACK_CUTS` with a Punzi figure of merit and saves them to `analyses/hnl_mupi/thresholds.txt`.

* The thresholds act on every PFP of a slice, so this notebook loads the **PFP-level** frames (`preprocess_*_pfp`, slice cuts applied) instead of the collapsed ones; per-PFP quantities are computed once, and every scan point only recounts per slice (`ana.tune`).
* The scans run in cut order. Each holds the cuts before it at their current thresholds, varies one threshold, and evaluates `eff_S / (a/2 + sqrt(B))` after its own cut (a = 3). The threshold is set to the loosest value within 2% of the FOM peak, then the next scan starts from it.
* Truth enters only through the HNL-matched slice label; every cut is reco-level. The slice cuts (`DEFAULT_CUTS`) are not tuned.
* After saving, reload the frames for the selection notebook with `preprocess_fn=partial(ana.preprocess_mcbnb, thresholds=ana.read_thresholds())` and `ana.track_cuts(t["open_angle_max"])`.

Heavy: run the load cells on the JupyterHub (or set `MAX_EVENTS` for a layout check).

In [ ]:
import sys
from pathlib import Path

sys.path[:0] = ["/exp/sbnd/app/users/sdas1/cafpyana", "/exp/sbnd/app/users/sdas1"]

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import cafpybara.analyses.hnl_mupi as ana
from cafpybara.analyses.hnl_mupi import config

PLOT_DIR = Path("/exp/sbnd/app/users/sdas1/hnl/plots/track_thresholds_v1")
PLOT_DIR.mkdir(parents=True, exist_ok=True)
MASSES = (300, 365)
REF_MASS = 300            # the mass whose FOM sets the thresholds
MAX_EVENTS = None         # e.g. 15000 for a quick layout check
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)

## 1. Load PFP-level frames (slice cuts applied) and build the tuning samples

In [ ]:
STREAM = True   # read the frames from dCache over xrootd (no /pnfs mount needed, e.g. on JupyterHub)
if STREAM:
    ana.ensure_token()      # asks for a token (prints a link to approve) if none is valid
    BNB_FILE = ana.xrootd_url(config.BNB_FILE)
    HNL_FILES = {m: ana.xrootd_url(config.HNL_FILES[m]) for m in MASSES}
else:
    BNB_FILE, HNL_FILES = config.BNB_FILE, config.HNL_FILES

In [ ]:
bnb_pfp, POT_BNB, _ = ana.load_mc(BNB_FILE, max_events=MAX_EVENTS,
                                  preprocess_fn=ana.preprocess_mcbnb_pfp, cuts=ana.DEFAULT_CUTS)
W = config.TARGET_POT / POT_BNB
S_BNB = ana.make_sample(bnb_pfp, W)
del bnb_pfp
S_HNL = {}
for m in MASSES:
    h, pot, _ = ana.load_mchnl(HNL_FILES[m], max_events=MAX_EVENTS,
                               preprocess_fn=ana.preprocess_mchnl_pfp, cuts=ana.DEFAULT_CUTS)
    S_HNL[m] = ana.make_sample(h, config.TARGET_POT / pot, hnl=True)
    del h
print(f"BNB: {len(S_BNB.slc)} slices after the slice cuts, weight {W:.3f}")
for m, s in S_HNL.items():
    print(f"m{m}: {len(s.slc)} slices, expected HNL-matched at {config.TARGET_POT:.0e} POT = {s.slc.w[s.slc.sig].sum():.1f}")

## 2. Scan and choose (starts from `thresholds.txt`)

In [ ]:
THR, TABLES = ana.tune(S_BNB, S_HNL, REF_MASS, a=3.0, tol=0.02)

## 3. Scan tables
Per value: BNB slices after the cut (`B`, `B_n`), HNL efficiency and Punzi FOM per mass; `chosen` marks the pick. Check `B_n` at the chosen point: a small number means the choice is statistics-limited.

In [ ]:
for p, t in TABLES.items():
    print(p); display(t.round(4))

## 4. Variable distributions with the Punzi FOM
Candidate PFPs of each cut (BNB stack, HNL line at the reference mass); the lower panel has the cumulative per-candidate FOM and, on the right axis, the slice-level FOM the threshold is set from (red line: chosen).

In [ ]:
for sc in ana.SCANS:
    fig = ana.plot_param(sc, TABLES[sc.param], THR, S_BNB, S_HNL, REF_MASS, W,
                         savefig=str(PLOT_DIR / f"tune_{sc.param}_m{REF_MASS}.pdf"))
    plt.show()

## 5. Save
Writes `analyses/hnl_mupi/thresholds.txt` (overwrites; commit it with the plots' provenance in the message).

In [ ]:
IF_SAVE = False
notes = {p: f"FOM peak {t[f'fom {REF_MASS}'].max():.3g}, B_n={int(t.B_n[t.chosen].iloc[0])}" for p, t in TABLES.items()}
display(pd.Series(THR))
if IF_SAVE:
    path = ana.write_thresholds(THR, header=f"Track-cut thresholds, tuned with the Punzi FOM (a=3), reference mass m{REF_MASS}, "
                                            f"{config.TARGET_POT:.0e} POT.\nGenerated by examples/hnl_mupi_thresholds.ipynb.", notes=notes)
    print(open(path).read())